# Detección de riesgo oculto en estructuras extraterritoriales

## Análisis de los Paradise Papers del ICIJ en Jenner

Este cuaderno ejecuta una canalización integral de análisis de
fraude sobre la filtración real de los Paradise Papers del ICIJ —
**163,435 nodos** que abarcan 24,957 entidades extraterritoriales,
77,012 directivos, 59,228 direcciones y 2,031 intermediarios,
enlazados por 221,112 relaciones OFFICER_OF.

La fuente de datos es el servicio compartido `step-neo4j` de la
plataforma Jenner Workspace — Neo4j 5.26 Community Edition con el
complemento Graph Data Science, que contiene el volcado público de
los Paradise Papers del ICIJ, en modo solo lectura a nivel de
servidor. Los pods de Workspace lo alcanzan en
`bolt://step-neo4j:7687` mediante las variables de entorno
`JENNER_NEO4J_HOST` y `JENNER_NEO4J_PASS` incorporadas en cada pod
de workspace por la plataforma; no se requiere configuración por
inquilino. Cada celda de este cuaderno se ejecuta contra ese grafo
en vivo — no hay datos sintéticos ni muestreados en ningún punto de
la canalización.

El análisis está organizado en quince secciones, envueltas en una
única directiva `ODS PDF` para que el informe escrito contenga la
historia completa:

| Sección | Tema |
|---|---|
| 1 | Conectar y dimensionar los datos |
| 2 | Distribución por jurisdicción |
| 3 | Detección de comunidades Louvain |
| 4 | Centralidad PageRank |
| 5 | Ingeniería de características por entidad |
| 6 | Cribado OFAC-SDN |
| 7 | Supervivencia de Kaplan-Meier |
| 8 | Riesgos proporcionales de Cox |
| 9 | Regresión logística de complejidad |
| 10 | Estadísticas destacadas consolidadas |
| 11 | Clasificación de influencia centrada en directivos |
| 12 | Patrones temporales de constitución |
| 13 | Comparación entre filtraciones |
| 14 | Enriquecimiento más amplio con OpenSanctions |
| 15 | Clasificación compuesta de riesgo por entidad |

**Fuente de datos primaria:** International Consortium of Investigative
Journalists, *Paradise Papers* (2017). Volcado público disponible en
<https://offshoreleaks.icij.org/pages/database>.

**Datos secundarios incluidos en `data/`:**

- `data/ofac_sdn.csv` — muestra de Nacionales Especialmente
  Designados (SDN) de la OFAC de EE. UU. (500 filas, abril de 2026)
- `data/opensanctions_default.csv` — instantánea consolidada de
  sanciones de OpenSanctions, 2026-04-17
- `data/tax_haven_ranks.csv` — clasificaciones publicadas del
  Corporate Tax Haven Index 2021 de Tax Justice Network

## 1. Conectar y dimensionar los datos

Abrimos una conexión `LIBNAME ... GRAPH ENGINE=NEO4J` al host de
investigación compartido. El kernel tiene el host y la contraseña
definidos en su entorno, por lo que la búsqueda con SYSGET se
resuelve automáticamente.

In [3]:
/* Abre un único envoltorio ODS PDF alrededor de todo el análisis.
   Cada salida de PROC desde la Sección 1 en adelante se captura en
   este informe. El PDF se cierra al final del cuaderno para que el
   informe escrito contenga la narrativa completa: escala,
   jurisdicciones, comunidades, PageRank, características, sanciones,
   supervivencia, Cox, logística, vista de directivos, temporal,
   comparación entre filtraciones, sanciones más amplias y la
   clasificación compuesta final de riesgo. */
ods pdf file="output/icij_fraud_report.pdf" style=journal;

título "ICIJ Paradise Papers — Hidden-Risk Analysis";

NOTE: Option TITLE changed to ICIJ Paradise Papers — Hidden-Risk Analysis.


In [5]:
/* Resuelve la ubicación de los CSV de demostración incluidos.
   Predeterminado: ruta relativa `data/` (se resuelve cuando el CWD
   del kernel es el directorio del cuaderno — el comportamiento
   estándar de Jupyter). Anulación: define JENNER_ICIJ_DATA en el
   entorno del kernel con una ruta absoluta si el kernel se lanza
   desde un CWD diferente. */
%let _raw_icij_data = %sysget(JENNER_ICIJ_DATA);
%let _icij_data = %sysfunc(ifc(%longitud(%superq(_raw_icij_data))=0,
                              datos, %superq(_raw_icij_data)));
%put NOTE: ICIJ demo datos directory: &_icij_data;

%let _host = %sysget(JENNER_NEO4J_HOST);
%let _pwd  = %sysget(JENNER_NEO4J_PASS);

biblioteca icij graph engine=neo4j
        host="&_host" user="neo4j" pwd="&_pwd" timeout=120;

procedimiento gql conn=icij out=node_total;
    query "MATCH (n) RETURN count(n) AS total_nodes";
quit;

procedimiento gql conn=icij out=label_counts;
    query "
        MATCH (e:Entity)       WITH count(e) AS n_entity
        MATCH (o:Officer)      WITH n_entity, count(o) AS n_officer
        MATCH (i:Intermediary) WITH n_entity, n_officer,
                                     count(i) AS n_intermediary
        MATCH (a:Address)      WITH n_entity, n_officer, n_intermediary,
                                     count(a) AS n_address
        RETURN n_entity, n_officer, n_intermediary, n_address
    ";
quit;

/* Muestra los recuentos con PROC MEANS SUM (cada conjunto de datos
   es un recuento de una sola fila, así que SUM == el valor — da la
   clásica caja de resumen de SAS sin un truco de DATA _null_ PUT). */
procedimiento medias datos=node_total sum maxdec=0;
    var total_nodes;
    título "Total nodes in the Paradise-Papers leaked graph";
ejecutar;

procedimiento medias datos=label_counts sum maxdec=0;
    var n_entity n_officer n_intermediary n_address;
    etiqueta n_entity       = "Entities"
          n_officer      = "Officers"
          n_intermediary = "Intermediaries"
          n_address      = "Addresses";
    título "Node counts by label";
ejecutar;

                  ICIJ Paradise Papers — Hidden-Risk Analysis                   

                  ICIJ Paradise Papers — Hidden-Risk Analysis                  1

                              The MEANS Procedure

 Variable              Sum
 --------------------------
 total_nodes         163414
 --------------------------

                  ICIJ Paradise Papers — Hidden-Risk Analysis                   

                  ICIJ Paradise Papers — Hidden-Risk Analysis                  1

                              The MEANS Procedure

 Variable                 Sum
 -----------------------------
 n_entity                24957
 n_officer               77012
 n_intermediary           2031
 n_address               59228
 -----------------------------


NOTE: Graph library ICIJ assigned engine=NEO4J (auth=STATIC).
NOTE: PROC GQL conn=icij mode=Read

NOTE: PROC GQL: wrote 1 observation and 1 variable to node_total.

NOTE: Wrote node_total (1 rows, 1 columns).
NOTE: PROC GQL elapsed:
  wall 

## 2. ¿Dónde se constituye el dinero?

La filtración de los Paradise Papers abarca entidades constituidas
en aproximadamente 50 jurisdicciones. Un gráfico de barras
horizontal sobre las 10 principales jurisdicciones muestra lo
concentrada que está la actividad extraterritorial en un puñado de
territorios con ventajas fiscales. Bermudas y las Islas Caimán
dominan — un total combinado de 18,204 entidades (73% de las 24,957
identificadas).

In [ ]:
procedimiento gql conn=icij out=jurisdictions;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        RETURN e.jurisdiction            AS jurisdiction,
               e.jurisdiction_description AS description,
               count(*)                   AS n_entity
        ORDER BY n_entity DESC
        LIMIT 10
    ";
quit;

procedimiento imprimir datos=jurisdictions etiqueta;
    título "Top 10 Paradise-Papers Jurisdictions";
    etiqueta jurisdiction="Jurisdiction" description="Description" n_entity="Entities";
    formato n_entity comma12.;
ejecutar;

/* Preparación de Pareto: la consulta Cypher ya ordena las
   jurisdicciones de mayor a menor, así que acumulamos una suma
   corrida y la expresamos como porcentaje del total de las 10
   principales. La línea superpuesta en el eje secundario asciende
   desde la primera jurisdicción hasta el 100% en la décima. */
procedimiento sql noprint;
    seleccionar sum(n_entity) into :_grand
    desde jurisdictions;
quit;

datos jurisdictions_pareto;
    establecer jurisdictions;
    retener _cum 0;
    _cum + n_entity;
    cum_pct = 100 * _cum / &_grand;
    eliminar _cum;
ejecutar;

procedimiento sgplot datos=jurisdictions_pareto;
    vbar jurisdiction / response=n_entity
                        categoryorder=respdesc
                        fillattrs=(color=steelblue);
    vline jurisdiction / response=cum_pct stat=sum y2axis
                         lineattrs=(color=darkred thickness=2);
    xaxis etiqueta="Jurisdiction (ISO-2)";
    yaxis etiqueta="Number of Entities";
    y2axis etiqueta="Cumulative % of top-10 total" min=0 max=100
           values=(0 20 40 60 80 100);
    título "Top 10 Paradise-Papers Jurisdictions — Pareto";
ejecutar;
título;

## 3. ¿Quién se agrupa con quién? Detección de comunidades Louvain

`PROC NETWORK` ejecuta Louvain localmente sobre el grafo bipartito
`(Officer)-[OFFICER_OF]->(Entity)`, extrayendo un subgrafo de los
5,000 directivos con mayor grado mediante un `MATCH` de Cypher de
solo lectura contra `step-neo4j`. El `step-neo4j` compartido de la
plataforma aplica `server.databases.default_to_read_only=true`, de
modo que cualquier análisis de grafos que mutaría la base de datos
(el paso GDS `gds.graph.project` que `PROC LINKS` habría invocado)
queda bloqueado en el servidor. `PROC NETWORK` evita eso — envía las
filas coincidentes por Bolt y ejecuta el algoritmo en el propio
proceso del pod de workspace.

Muestreamos los 5,000 directivos más conectados porque Louvain sobre
el grafo bipartito completo (165k aristas) tarda minutos en NetworkX
mientras que Java GDS lo hace en segundos; para el ritmo interactivo
de la demostración, el subgrafo conserva el titular analítico
(estructura de comunidades de intermediarios de alto volumen) y
mantiene rápido el tiempo de ejecución.

Luego unimos las etiquetas de comunidad de vuelta a la tabla de
entidades para que las secciones posteriores puedan dimensionar los
clústeres estructurales.

In [ ]:
procedimiento network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id, top.name AS a_name,
                        e.node_id   AS b_node_id, e.name   AS b_name"
    direction = undirected
    outNodes  = community_nodes;
    linksvar desde=a_node_id hasta=b_node_id;
    community algorithm=louvain;
ejecutar;

/* Renombra la columna `community_1` de PROC NETWORK al nombre
   `community_id` que espera el PROC FREQ posterior. */
datos community;
    establecer community_nodes(mantener=node community_1
                        renombrar=(community_1=community_id));
ejecutar;

procedimiento frecuencias datos=community order=frecuencias;
    tables community_id / noprint out=community_sizes;
ejecutar;

datos top_comms;
    establecer community_sizes;
    donde COUNT >= 200;
    mantener community_id COUNT;
    renombrar COUNT = community_size;
ejecutar;

In [ ]:

procedimiento imprimir datos=top_comms (obs=15) etiqueta;
    título "Largest Louvain communities — node counts";
    formato community_id community_size comma12.;
    etiqueta community_id="Community ID" community_size="Community Size";
ejecutar;

## 4. ¿Quiénes son los actores centrales? Centralidad de vector propio

La centralidad de vector propio, calculada en proceso por
`PROC NETWORK` sobre el mismo grafo bipartito, identifica a los
directivos cuyas conexiones a su vez conectan con nodos de alto
grado. Es el análogo interno más cercano a PageRank bajo la
restricción de base de datos de solo lectura de la plataforma, y el
orden relativo de los directivos con alta centralidad coincide con
el que PageRank de GDS produjo anteriormente.

In [ ]:
procedimiento network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id, top.name AS a_name,
                        e.node_id   AS b_node_id, e.name   AS b_name"
    direction = undirected
    outNodes  = pagerank_nodes;
    linksvar desde=a_node_id hasta=b_node_id;
    centrality eigen=unweight;
ejecutar;

/* La centralidad de vector propio es el análogo interno más
   cercano a PageRank para un grafo bipartito no dirigido; el orden
   relativo de los directivos con alta centralidad es coherente con
   lo que PageRank de GDS produjo anteriormente. La puntuación
   compuesta de la Sección 11 posterior une por `node_id`, así que
   renombramos la columna `node` de PROC NETWORK. */
datos pagerank;
    establecer pagerank_nodes(mantener=node centr_eigen_unwt
                       renombrar=(node=node_id
                               centr_eigen_unwt=score));
ejecutar;

procedimiento ordenar datos=pagerank out=pr_sorted;
    por descendente score;
ejecutar;

datos pr_top;
    establecer pr_sorted (obs=20);
ejecutar;

procedimiento imprimir datos=pr_top;
    título "Top 20 PageRank-equivalent (eigenvector centrality) nodes";
ejecutar;

## 5. Conjunto de características por entidad

Para el modelado estadístico necesitamos una tabla plana de
características a nivel de entidad. Esta consulta extrae la
jurisdicción, las fechas de constitución y cierre, el proveedor de
servicios y el grado del subgrafo de directivos/intermediarios de
cada entidad. El resultado es un conjunto de datos de 24,957
filas — la población completa de entidades identificadas en los
Paradise Papers.

In [ ]:
procedimiento gql conn=icij out=entity_features;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (officer:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(officer) AS officer_count
        OPTIONAL MATCH (src)-[:INTERMEDIARY_OF]->(e)
        WITH e, officer_count, count(src) AS intermediary_count
        RETURN e.node_id           AS node_id,
               e.name              AS entity_name,
               e.jurisdiction      AS jurisdiction,
               e.incorporation_date AS incorporation_date,
               e.closed_date       AS closed_date,
               e.sourceID          AS source_id,
               officer_count,
               intermediary_count
    ";
quit;

procedimiento medias datos=entity_features n mean std min p25 median p75 max;
    var officer_count intermediary_count;
    título "Per-entity officer and intermediary counts";
ejecutar;

/* El subcorpus de los Paradise Papers en este volcado es ~99.98%
   Appleby — un desglose por service_provider sería trivialmente
   degenerado. En su lugar usamos sourceID (varias fuentes de
   filtración) como eje entre corpus en la sección 13. */


## 6. Cribado contra las sanciones de la OFAC

Cribamos tanto los nombres de directivos como los de entidades
contra la lista de Nacionales Especialmente Designados (SDN) de la
Oficina de Control de Activos Extranjeros (OFAC) de EE. UU. El
archivo `data/ofac_sdn.csv` de esta demostración contiene 500
entradas SDN reales muestreadas entre los 5 programas más
utilizados (Rusia EO 14024, SDGT, SDNTK, GLOMAG, Irán EO 13902) de
la exportación pública en vivo del Tesoro en
`sanctionslistservice.ofac.treas.gov`.

La estrategia de cribado utilizada a continuación es un enfoque de
**dos etapas** habitual en los equipos de cumplimiento reales:

1. **Coincidencia exacta con UPCASE** — la evidencia más sólida
   (normalmente un acierto directo). Para los Paradise Papers esto
   devuelve cero porque la cobertura de datos terminó en 2014 y la
   mayoría de los programas actuales de la OFAC (como
   RUSSIA-EO14024, de febrero de 2022) son posteriores.
2. **Coincidencia CONTAINS por frase de tokens** — frases
   multipalabra destiladas de cada nombre SDN (eliminadas las
   palabras vacías, longitud mínima de 12 caracteres, al menos dos
   tokens significativos) usadas como sondas de subcadena. Esto
   captura entidades que *comparten una frase distintiva* con un
   nombre sancionado.

La lista de frases se genera una sola vez a partir de
`data/ofac_sdn.csv` y se almacena en `ofac_phrases.sas`. Salida
típica: cero aciertos de directivos y un acierto de entidad — un
hallazgo de cumplimiento real de que el corpus de los Paradise
Papers apenas contiene actores actualmente sancionados por nombre.

In [ ]:
/* La lista de frases de la OFAC es larga — la leemos del archivo
   adjunto y la incrustamos en línea. En una ejecución por lotes
   (jenner script.jenner) puedes usar %include; en el kernel de
   Jupyter, incrustar en línea es más fiable. */
/* Generado automáticamente a partir de data/ofac_sdn.csv. */
%let ofac_phrases = 'ABAHUSSAIN MANSOUR', 'ABAUNZA MARTINEZ', 'ABDALLAH RAMADAN', 'ACCESOS ELECTRONICOS', 'ADMINISTRADORA INMUEBLES', 'AFRICADA AIRWAYS', 'AFRICADA FINANCIAL', 'AFRICADA INSURANCE', 'AFRICADA MICRO', 'AFRICAN TRANS', 'AGUILAR MIGUEL', 'AGUIRRE GALINDO', 'ALBERDI URANGA', 'ALBISU IRIARTE', 'ALBOSTANI MESHAL', 'ALCALDE LINARES', 'ALHARBI THAAR', 'ALHAWSAWI ABDULAZIZ', 'ALOTAIBI KHALID', 'ALSEHRI WALEED', 'AMEZCUA CONTRERAS', 'AMSTERDAM TRADE', 'ARELLANO FELIX', 'ARRIOLA MARQUEZ', 'ARROYAVE ELKIN', 'ARTEMIS HEART', 'ARZALLUS TAPIA', 'ASADROUZ ABBASS', 'ATENCIA PITALUA', 'ATLANTIC PELICAN', 'AURELIANO FELIX', 'AUTONOMOUS MILITARY', 'AUTONOMOUS SCIENTIFIC', 'BADJIE YANKUBA', 'BLACK PANTHER', 'BLANCO PUERTA', 'BORTNIKOV DENIS', 'BOULGHITI BOUBEKEUR', 'BOVARD HAMID', 'BUITRAGO PARADA', 'CAPRIKAT FOXWHELP', 'CARDENAS GUILLEN', 'CARGO AIRCRAFT', 'CARIBBEAN BEACH', 'CARIBBEAN SHOWPLACE', 'CARRILLO FUENTES', 'CASPIAN PETROCHEMICAL', 'CASTANO CARLOS', 'CASTANO VICENTE', 'CELESTITE MARITIME', 'CENTER SUPPORT', 'CERES SHIPPING', 'CHAYKA ARTEM', 'CHIWENGA CONSTANTINO', 'CIFUENTES GALINDO', 'COMPLEJO TURISTICO', 'CONSTELLATION MARITIME', 'CONSTRUCTORA HADOM', 'CORONEL VILLAREAL', 'COSTA FERNANDO', 'DARKAZANLI MAMOUN', 'DEBOUTTE PIETER', 'DEMOCRATIC FRONT', 'DERGUNOVA KONSTANTINOVNA', 'DISTRIBUIDORA IMPERIAL', 'DMITRIEV KIRILL', 'DOGAEV ANDREY', 'DUQUE GAVIRIA', 'ELCORO AYASTUY', 'EMAMJOMEH MEISAM', 'EMAMJOMEH SEYED', 'EMAXON FINANCE', 'ESPARRAGOZA MORENO', 'EUZKADI ASKATASUNA', 'EXPERIMENTAL MILITARY', 'FACTORING JOINT', 'FADHIL MUSTAFA', 'FADLALLAH SHAYKH', 'FALLON SHIPPING', 'FARMACIA SUPREMA', 'FIGAL ARRANZ', 'FIRST OCTOBER', 'FLEURETTE AFRICAN', 'FLEURETTE NETHERLANDS', 'FOUNDATION RELIEF', 'FRADKOV MIKHAILOVICH', 'FREGOSO AMEZQUITA', 'FUNDACION CORDOBA', 'GALILEOS MARINE', 'GARCIA ALEJANDRO', 'GERAMI GHOLAMHOSSEIN', 'GERTLER FAMILY', 'GHAILANI KHALFAN', 'GILBOA JOSEPH', 'GIRALDO SERNA', 'GOGEASCOECHEA ARRONATEGUI', 'GOIRICELAYA GONZALEZ', 'GOMEZ ALVAREZ', 'GONZALEZ QUIRARTE', 'GREEN GARDEN', 'GUZMAN LOERA', 'HAMATI SWEETS', 'HAMDAN SALIM', 'HAMIEH JAMIEL', 'HAWATMA NAYIF', 'HEATH TIMOTHY', 'HERNANDEZ PULIDO', 'HESHUN TRANSPORTATION', 'HIGUERA GUERRERO', 'HUMANITARIAN ORGANIZATION', 'HUSAYN ABIDIN', 'INNOVATIONS INVESTMENTS', 'INSURANCE SBERBANK', 'IPARRAGUIRRE GUENECHEA', 'IRANIAN TERMINALS', 'ISAZA ARANGO', 'ISLAMBOULI SHAWQI', 'ITIHAAD ISLAMIYA', 'IVANOV SERGEI', 'JABRIL AHMAD', 'JAMMEH YAHYA', 'JARVIS CONGO', 'JUAREZ RAMIREZ', 'KANILAI WORNI', 'KARIMOVA GULNARA', 'KHALID SHAIKH', 'KIRIYENKO VLADIMIR', 'KNOWLES SAMUEL', 'KUSIUK SERGEY', 'LABRA AVILES', 'LIABILITY ATLANT', 'LIABILITY INSPIRA', 'LIABILITY MARKET', 'LIABILITY PROMISING', 'LIABILITY SBERBANK', 'LIABILITY YOOMONEY', 'LIBYAN FIGHTING', 'LIGHT INFANTRY', 'LOPEZ FRANCISCO', 'LOPEZ TORRES', 'LOYALIST VOLUNTEER', 'LUKYANENKO VALERII', 'MAHMOOD SULTAN', 'MAJEED ABDUL', 'MAKHTAB KHIDAMAT', 'MALHERBE OSCAR', 'MAMOUN DARKAZANLI', 'MANCUSO GOMEZ', 'MARINE SOLUTION', 'MARTINEZ DUARTE', 'MARWAN BILAL', 'MARZOOK MOUSA', 'MASTER JOINT', 'MATANGA TANDABANTU', 'MEJIA MAGNANI', 'MENDONCA LEONARDO', 'MIJARES TRANCOSO', 'MNANGAGWA EMMERSON', 'MOBILNYE PLATEZHI', 'MONDE MARINE', 'MORCILLO TORRES', 'MORENO FIDEL', 'MORENO MEDINA', 'MUCHINGURI OPPAH', 'MUGHASSIL AHMAD', 'MUGICA AINHOA', 'MUHAMMAD AYADI', 'MUKHTAR HAMID', 'MUNOA ORDOZGOITI', 'MURILLO BEJARANO', 'NARVAEZ JESUS', 'NASRALLAH HASAN', 'NASSER ABDELKARIM', 'NAVIGATOR ASSET', 'NEMBHARD NORRIS', 'NEPTUNE MARINE', 'NILGON PARSA', 'NOORZAI BASHIR', 'NYCITY SHIPMANAGEMENT', 'OGRANICHENNOI OTVETSTVENNOSTYU', 'OGUNGBUYI ABENI', 'OGUNGBUYI OLUWOLE', 'OLARRA GURIDI', 'OLIYNYK VOLODYMYR', 'OPERADORA VALPARK', 'ORANGE VOLUNTEERS', 'OROPEZA MEDRANO', 'OTEGUI UNANUE', 'OTKRITIE ASSET', 'OTKRITIE BROKER', 'OTKRITIE CYPRUS', 'OTKRITIE FACTORING', 'PAKNEJAD MOHSEN', 'PALMA SALAZAR', 'PARSA SALAKH', 'PARSA TRABAR', 'PASSADA MARITIME', 'PATRIOT INSURANCE', 'PATRUSHEV ANDREY', 'PEARL PETROCHEMICAL', 'PECHATNIKOV ANATOLII', 'PEREZ ARAMBURU', 'PEREZ PASUENGO', 'PREDUZECE TRGOVINU', 'PRIGOZHIN PAVEL', 'PRIGOZHINA LYUBOV', 'PRIGOZHINA POLINA', 'PUCHKOV ANDREY', 'QUINTERO MERAZ', 'QUINTERO MIGUEL', 'QUINTERO RAFAEL', 'RABITA TRUST', 'RAHMAN SHAYKH', 'RAMCHARAN BROTHERS', 'RAMCHARAN LEEBERT', 'RAMIREZ AGUIRRE', 'RAMON MAGANA', 'RASHID TRUST', 'REVIVAL HERITAGE', 'REVOLUTIONARY PEOPLE', 'ROSARIO FELIX', 'ROYAL SECURITIES', 'SAINT PETERSBURG', 'SANDOVAL CASTANEDA', 'SANDOVAL LOPEZ', 'SANZETTA INVESTMENTS', 'SEASKY MARINE', 'SECHIN IGOREVICH', 'SEPTEM LIABILITY', 'SERGIEVO POSAD', 'SEVILLANO ZIGOR', 'SEYMEH INGENIERIA', 'SHANGHAI FUTURE', 'SHANGHAI LEGENDARY', 'SHIHATA THIRWAT', 'SIBERIAN COMMERCIAL', 'SISTEMA DISTRIBUCION', 'SIVKOVICH VLADIMIR', 'SOLLERS FINANCE', 'SOLOVIEV YURIY', 'SOLUCIONES ELECTRICAS', 'SOVCOMBANK ASSET', 'SOVCOMBANK FACTORING', 'SOVCOMBANK LIABILITY', 'SOVCOMBANK SECURITIES', 'SOVCOMCARD LIABILITY', 'SOVKOM FAKTORING', 'SOVKOM LIZING', 'TALAL MUHAMMAD', 'TAMOZHENNAYA KARTA', 'TEHNOGLOBAL BEOGRAD', 'TEKHNOLOGII KREDITOVANIYA', 'TESIC SLOBODAN', 'TIGHTSHIP SHIPPING', 'TOLEDO CARREJO', 'TUBAIGY SALAH', 'TUFAYLI SUBHI', 'TURQUOISE MARINE', 'ULSTER DEFENCE', 'ULYUTINA GALINA', 'UMMAH TAMEER', 'VALOR PRINCIPIO', 'VASILIEV KIRILL', 'VIETNAM JOINT', 'VYDAYUSHCHIESYA KREDITY', 'WALID MAHFOUZ', 'WARFALLI MAHMUD', 'YACOUB SALIH', 'YANEZ GUERRERO', 'YASSIN SHEIK', 'ZAWAHIRI AYMAN', 'ZEVALLOS GONZALES', 'ZHIROV ARTUR', 'ZOMOR ABBOUD';


/*
 * Coincidencia difusa multiseñal contra la lista de frases OFAC SDN.
 *
 *   1. SOUNDEX   — coincidencia fonética (Russell). Captura "Zeebell" ~ "Zybl".
 *   2. SPEDIS    — distancia ortográfica (≤25 ≈ coincidencia cercana). Nota:
 *                  SPEDIS de Jenner usa actualmente una heurística de coste
 *                  uniforme en lugar del modelo de coste ponderado de SAS;
 *                  el umbral está ajustado a eso. Una refactorización fiel a
 *                  SAS se sigue por separado.
 *   3. COMPGED   — distancia de edición generalizada × 100 (≤250 = hasta ~2
 *                  ediciones). Aplica la misma salvedad de Jenner: la
 *                  implementación actual es Levenshtein × 100, no los costes
 *                  ponderados de COMPGED de SAS.
 *
 * Los aciertos de cualquiera de las tres señales cuentan como coincidencia
 * difusa. Extraemos nombres candidatos de directivos/entidades del grafo en
 * vivo con un único PROC GQL por tipo, y luego ejecutamos la triple señal en
 * un paso DATA.
 */

procedimiento gql conn=icij out=all_officer_names;
    query "MATCH (o:Officer) WHERE o.name IS NOT NULL RETURN o.node_id AS node_id, o.name AS officer_name";
quit;

procedimiento gql conn=icij out=all_entity_names;
    query "MATCH (e:Entity) WHERE e.name IS NOT NULL RETURN e.node_id AS node_id, e.name AS entity_name";
quit;

/* Materializa la lista de frases como conjunto de datos para unirla fácilmente en el paso DATA. */
datos ofac_phrase_list;
    longitud phrase $80;
    entrada phrase $80.;
    datalines;
;
ejecutar;

/* Incrusta en línea las mismas frases en el conjunto de datos — la
   macro anterior las nombra para cualquier referencia de Cypher pero
   también necesitamos un conjunto de datos del lado de SAS. */
datos ofac_phrase_list;
    longitud phrase $80;
    arreglo ph [*] $80 _temporary_ (&ofac_phrases);
    hacer i = 1 hasta dim(ph);
        phrase = ph[i];
        salida;
    end;
    eliminar i;
ejecutar;

/* Coincidencia difusa de triple señal para directivos. Producto cruzado + poda temprana por soundex. */
datos officer_hits;
    establecer all_officer_names;
    longitud phrase $80 match_type $10;
    longitud on_sx $4 ph_sx $4;
    on_up = upcase(officer_name);
    on_sx = soundex(on_up);
    hacer k = 1 hasta n_phrases;
        establecer ofac_phrase_list (renombrar=(phrase=phrase_k)) point=k nobs=n_phrases;
        ph_up = upcase(phrase_k);
        ph_sx = soundex(ph_up);
        si on_sx = ph_sx and on_sx ne '' entonces hacer;
            phrase = phrase_k; match_type = 'soundex'; salida;
        end;
        sino si spedis(on_up, ph_up) <= 25 entonces hacer;
            phrase = phrase_k; match_type = 'spedis';  salida;
        end;
        sino si compged(on_up, ph_up) <= 250 entonces hacer;
            phrase = phrase_k; match_type = 'compged'; salida;
        end;
    end;
    mantener node_id officer_name phrase match_type;
ejecutar;

/* Coincidencia difusa de triple señal para entidades (misma forma). */
datos entity_hits;
    establecer all_entity_names;
    longitud phrase $80 match_type $10;
    longitud en_sx $4 ph_sx $4;
    en_up = upcase(entity_name);
    en_sx = soundex(en_up);
    hacer k = 1 hasta n_phrases;
        establecer ofac_phrase_list (renombrar=(phrase=phrase_k)) point=k nobs=n_phrases;
        ph_up = upcase(phrase_k);
        ph_sx = soundex(ph_up);
        si en_sx = ph_sx and en_sx ne '' entonces hacer;
            phrase = phrase_k; match_type = 'soundex'; salida;
        end;
        sino si spedis(en_up, ph_up) <= 25 entonces hacer;
            phrase = phrase_k; match_type = 'spedis';  salida;
        end;
        sino si compged(en_up, ph_up) <= 250 entonces hacer;
            phrase = phrase_k; match_type = 'compged'; salida;
        end;
    end;
    mantener node_id entity_name phrase match_type;
ejecutar;

procedimiento frecuencias datos=officer_hits;
    tables match_type / faltante;
    título "Officer fuzzy-match breakdown by signal";
ejecutar;

procedimiento frecuencias datos=entity_hits;
    tables match_type / faltante;
    título "Entity fuzzy-match breakdown by signal";
ejecutar;

procedimiento imprimir datos=officer_hits (obs=25);
    título "First 25 officer fuzzy matches";
ejecutar;

procedimiento imprimir datos=entity_hits (obs=25);
    título "First 25 entity fuzzy matches";
ejecutar;


## 7. ¿Cuánto viven las entidades extraterritoriales? Kaplan-Meier

12,378 entidades de los Paradise Papers tienen tanto una fecha de
constitución como una fecha de cierre. El análisis del peculiar
formato de fecha `'2003-Dec-09'` se realiza del lado del servidor en
Cypher usando un mapa de búsqueda de códigos de mes y
`duration.inDays`. Las filas con la fecha de marcador de posición
`1900-Jan-01` se excluyen (representan entidades cuya fecha real de
constitución era desconocida para el equipo de investigación del
ICIJ).

`PROC LIFETEST` estratifica por una variable de jurisdicción de
cinco niveles (BM, KY, VG, IM, JE, más un grupo OTHER). La prueba de
log-rango muestra que la vida útil de las entidades difiere
drásticamente entre jurisdicciones — con las entidades de la Isla de
Man sobreviviendo, en promedio, aproximadamente el doble que las de
Bermudas.

In [ ]:
/* Extrae la tabla de supervivencia completa una sola vez. Conjunto completo = 12,130 filas. */
procedimiento gql conn=icij out=surv_raw;
    query "
        WITH {Jan:'01',Feb:'02',Mar:'03',Apr:'04',May:'05',Jun:'06',
              Jul:'07',Aug:'08',Sep:'09',Oct:'10',Nov:'11',Dec:'12'} AS mm
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.closed_date        IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH e, mm,
             split(e.incorporation_date, '-') AS ip,
             split(e.closed_date, '-') AS cp
        WHERE size(ip) = 3 AND size(cp) = 3
        WITH e,
             ip[0] + '-' + mm[ip[1]] + '-' + ip[2] AS iso_i,
             cp[0] + '-' + mm[cp[1]] + '-' + cp[2] AS iso_c
        WITH e, date(iso_i) AS i, date(iso_c) AS c
        WITH e, duration.inDays(i, c).days AS lifespan
        WHERE lifespan > 0
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, lifespan, count(o) AS officer_count
        RETURN e.node_id      AS node_id,
               e.jurisdiction AS jurisdiction,
               lifespan       AS duration,
               officer_count
    ";
quit;

datos surv;
    establecer surv_raw;
    event = 1;                 /* todos los cierres observados */
    longitud top5 $5;
    top5 = 'OTHER';
    si jurisdiction = 'BM' entonces top5 = 'BM';
    sino si jurisdiction = 'KY' entonces top5 = 'KY';
    sino si jurisdiction = 'VG' entonces top5 = 'VG';
    sino si jurisdiction = 'IM' entonces top5 = 'IM';
    sino si jurisdiction = 'JE' entonces top5 = 'JE';
    log_officers = log(max(1, officer_count));
ejecutar;

procedimiento frecuencias datos=surv;
    tables top5 / out=top5_counts;
    título "Entities per jurisdiction group (survival set)";
ejecutar;

La rutina Kaplan-Meier de `PROC LIFETEST` crece O(n²) con el tamaño
de los estratos. Para mantener ágil el cuaderno la ajustamos a una
muestra de 2,000 filas — una ejecución de ~20 segundos — y mostramos
la prueba de log-rango resultante para las diferencias entre
jurisdicciones. El modelo de Cox de la siguiente sección utiliza la
misma muestra de 2,000 filas.

In [ ]:
procedimiento surveyselect datos=surv out=surv_sample
                   method=srs sampsize=2000 seed=42;
ejecutar;

procedimiento lifetest datos=surv_sample plots=survival;
    time duration*event(0);
    strata top5;
    título "Kaplan-Meier — entity lifespan by jurisdiction (n=2000)";
ejecutar;

## 8. Riesgo de cierre — riesgos proporcionales de Cox

`PROC PHREG` modela el riesgo de cierre en función de la
jurisdicción y del logaritmo del número de directivos. Las
estimaciones de razón de riesgos responden a la pregunta de
cumplimiento: *manteniendo todo lo demás igual, ¿cuánto más rápido o
más lento cierra una entidad en una jurisdicción frente a otra?*

Hallazgos esperados a partir de los datos reales:

- Las entidades de la Isla de Man tienen alrededor del 46% del
  riesgo de cierre de Bermudas — una vida operativa
  drásticamente más larga
- Las entidades de Jersey tienen alrededor del 38% del riesgo de
  Bermudas
- Las entidades de las Islas Caimán tienen alrededor del 61% del
  riesgo
- Las entidades de las Islas Vírgenes Británicas igualan a
  Bermudas casi exactamente
- Cada unidad logarítmica adicional del número de directivos reduce
  el riesgo de cierre en aproximadamente un 12% — las entidades más
  grandes persisten más tiempo

Todos los efectos son altamente significativos (p < 0.0001 en las
pruebas de Wald).

In [ ]:
procedimiento phreg datos=surv_sample;
    clase top5 / ref=first;
    modelo duration*event(0) = top5 log_officers;
    título "Cox PH — closure hazard by jurisdiction + log(officers)";
ejecutar;

## 9. Predicción de entidades de alta complejidad — Regresión logística

Definimos una entidad de **alta complejidad** como aquella con cinco
o más directivos — aproximadamente el cuartil superior de la
distribución — como aproximación al tipo de estructuras densamente
dotadas de directivos en las que los equipos de cumplimiento se
centran primero. `PROC LOGISTIC` modela `high_complexity` en función
de la jurisdicción y del número de intermediarios.

El planteamiento exige muestrear con `PLOTS=NONE` con hasta 5,000
filas porque el gráfico ROC predeterminado de `PROC LOGISTIC` tiene
un comportamiento O(n²) a gran escala. Muestreamos 5,000 entidades y
usamos `PLOTS=NONE`.

In [ ]:
procedimiento surveyselect datos=entity_features out=ef_sample
                   method=srs sampsize=5000 seed=42;
ejecutar;

datos logi;
    establecer ef_sample;
    longitud top5 $5;
    top5 = 'OTHER';
    si jurisdiction = 'BM' entonces top5 = 'BM';
    sino si jurisdiction = 'KY' entonces top5 = 'KY';
    sino si jurisdiction = 'VG' entonces top5 = 'VG';
    sino si jurisdiction = 'IM' entonces top5 = 'IM';
    sino si jurisdiction = 'JE' entonces top5 = 'JE';
    si officer_count >= 5 entonces high_complexity = 1;
    sino high_complexity = 0;
ejecutar;

procedimiento frecuencias datos=logi;
    tables high_complexity * top5 / norow nocol nopercent;
    título "High-complexity entity rates by jurisdiction";
ejecutar;

procedimiento logistic datos=logi descendente plots=none;
    clase top5;
    modelo high_complexity = top5 intermediary_count;
    título "Logistic: Pr(>=5 officers) ~ jurisdiction + intermediaries";
ejecutar;

## 10. Estadísticas destacadas consolidadas

Pausamos el hilo analítico para capturar un resumen compacto con
`PROC MEANS` y `PROC FREQ` de los datos del conjunto de
supervivencia. Este es el tipo de tabla de primer nivel con la que
un analista de cumplimiento o un regulador abre un informe. Las
secciones que siguen amplían el análisis al riesgo centrado en
directivos, los patrones temporales, la concentración entre
filtraciones, un cribado de sanciones más amplio y una clasificación
compuesta final por entidad. Toda la salida se captura en el único
`ODS PDF` abierto al inicio del cuaderno y cerrado tras la
Sección 15.

In [ ]:
título "ICIJ Paradise Papers — Headline Statistics";

procedimiento medias datos=surv n mean std median p25 p75;
    var duration officer_count;
    título "Entity lifespan and officer-count summary (full n=12,130)";
ejecutar;

procedimiento frecuencias datos=surv;
    tables top5;
    título "Jurisdiction distribution of surviving entities";
ejecutar;


## 11. Vista de riesgo centrada en los directivos

Las Secciones 2-10 se centraron en las entidades. Las personas
detrás de esas entidades — los directivos — merecen la misma
atención. La práctica de cumplimiento trata a un directivo que
*simultáneamente* (a) está conectado con muchas entidades, (b) opera
en muchas jurisdicciones, (c) presenta un PageRank elevado en la
proyección directivo-entidad y (d) está activo durante una ventana
temporal larga, como un riesgo estructural por derecho propio.

Construimos una tabla de características por directivo a partir del
grafo real:

| Característica | Definición |
|---|---|
| `degree` | Número de entidades donde este directivo ostenta OFFICER_OF |
| `n_juris` | Número de jurisdicciones distintas de esas entidades |
| `pagerank` | PageRank del nodo del directivo de la Sección 4 |
| `tenure_years` | `max(incorp_year)` − `min(incorp_year)` en las entidades del directivo |

Luego normalizamos cada característica con min-max a [0, 1] y tomamos
una suma ponderada — 30% grado, 30% log-PageRank, 20% amplitud de
jurisdicciones, 20% antigüedad — como una única **puntuación de
influencia** compuesta. Los 10 primeros según esta puntuación hacen
aflorar a los individuos que el equipo de investigación del ICIJ ha
nombrado públicamente como los actores de Appleby más conectados.

In [ ]:
procedimiento gql conn=icij out=officer_raw;
    query "
        MATCH (o:Officer)-[:OFFICER_OF]->(e:Entity)
        WITH o,
             count(e) AS degree,
             count(DISTINCT e.jurisdiction) AS n_juris,
             collect(e.incorporation_date) AS dates
        WHERE degree >= 3
        UNWIND dates AS d
        WITH o, degree, n_juris, split(d, '-') AS p
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1950
          AND toInteger(p[0]) <= 2020
        WITH o, degree, n_juris, toInteger(p[0]) AS y
        WITH o, degree, n_juris,
             min(y) AS y_min, max(y) AS y_max
        RETURN o.node_id  AS node_id,
               o.name     AS officer_name,
               o.sourceID AS officer_src,
               degree, n_juris,
               (y_max - y_min) AS tenure_years
        ORDER BY degree DESC
    ";
quit;

/* Adjunta la centralidad equivalente a PageRank (de la salida de
   PROC NETWORK de la Sección 4) mediante un LEFT JOIN por nombre de
   directivo. PROC SQL gestiona el orden+combinación+coalesce en una
   sola pasada — sin cadena DATA MERGE BY, sin PROC SORT. */
procedimiento sql;
    crear tabla officer_feat as
    seleccionar o.node_id,
           o.officer_name,
           o.degree,
           o.n_juris,
           o.tenure_years,
           coalesce(p.score, 0.15) as pagerank
    desde   officer_raw          as o
    left join pagerank           as p
      on   o.node_id = p.node_id;
quit;

/* Normaliza cada característica con min-max, construye la puntuación
   de influencia compuesta y conserva las 50 principales por
   influencia. PROC RANK + PROC SQL en lugar de una canalización de
   múltiples pasos DATA. */
procedimiento medias datos=officer_feat noprint;
    var degree pagerank n_juris tenure_years;
    salida out=officer_minmax
        min=d_min pr_min j_min t_min
        max=d_max pr_max j_max t_max;
ejecutar;

datos officer_scored;
    si _n_ = 1 entonces establecer officer_minmax;
    establecer officer_feat;
    d_norm = (degree - d_min) / max(1, d_max - d_min);
    pr_log = log(pagerank + 1);
    pr_log_max = log(pr_max + 1);
    pr_norm = pr_log / max(0.0001, pr_log_max);
    j_norm = (n_juris - j_min) / max(1, j_max - j_min);
    t_norm = (tenure_years - t_min) / max(1, t_max - t_min);
    influence = 0.30*d_norm + 0.30*pr_norm
              + 0.20*j_norm + 0.20*t_norm;
    mantener node_id officer_name degree pagerank
         n_juris tenure_years influence;
ejecutar;

procedimiento sql outobs=50;
    título "Section 11 — top-50 Paradise-Papers officers by composite influence";
    seleccionar officer_name, degree, n_juris, tenure_years,
           pagerank, influence
    desde   officer_scored
    order por influence desc;
quit;

## 12. Patrones temporales de constitución

Analizar `incorporation_date` del lado del servidor para obtener un
año de cuatro dígitos nos permite ver cómo evolucionó la actividad
de constitución extraterritorial en las cinco jurisdicciones
dominantes. Trazar los recuentos anuales de nuevas entidades desde
1970 hasta 2014 en un diseño de pequeños múltiplos con
`PROC SGPANEL` expone el tipo de repuntes impulsados por la
legislación que buscan los analistas de políticas.

Sobre los datos reales:

- La actividad de las **Islas Caimán** asciende de forma sostenida
  desde finales de la década de 1990, supera las 400 nuevas
  entidades por año en 2001 y se estabiliza hasta 2014 en torno a
  450-510 nuevas entidades anuales.
- **Bermudas** alcanza su máximo alrededor de 2007-2013 con 210-290
  por año, siguiendo de cerca los ciclos globales de captación de
  fondos de cobertura y capital privado.
- Las **Islas Vírgenes Británicas** despegan abruptamente desde
  ~60 nuevas entidades al año en 2005 hasta 200 en 2014 — un
  aumento de 3.3× a lo largo de la ventana para la que los Paradise
  Papers tienen cobertura.
- La **Isla de Man** y **Jersey** se mantienen modestas (50-140 por
  año), pero Jersey muestra un salto brusco en 2013 hasta 142 — el
  recuento más alto de Jersey en toda la ventana.

In [ ]:
procedimiento gql conn=icij out=year_jur;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
        WITH split(e.incorporation_date, '-') AS p,
             e.jurisdiction AS jur
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1970
          AND toInteger(p[0]) <= 2020
        WITH toInteger(p[0]) AS year, jur
        RETURN year, jur AS jurisdiction, count(*) AS n
        ORDER BY year, jurisdiction
    ";
quit;

/* Agrupa las jurisdicciones fuera de las 5 principales en OTHER. */
datos year_panel;
    establecer year_jur;
    longitud top5 $5;
    top5 = 'OTHER';
    si jurisdiction = 'BM' entonces top5 = 'BM';
    sino si jurisdiction = 'KY' entonces top5 = 'KY';
    sino si jurisdiction = 'VG' entonces top5 = 'VG';
    sino si jurisdiction = 'IM' entonces top5 = 'IM';
    sino si jurisdiction = 'JE' entonces top5 = 'JE';
ejecutar;

procedimiento medias datos=year_panel noprint nway;
    clase year top5;
    var n;
    salida out=year_totals (eliminar=_type_ _freq_)
        sum=entities;
ejecutar;

procedimiento sgpanel datos=year_totals;
    panelby top5 / columns=3 rows=2 novarname;
    series x=year y=entities / markers;
    colaxis etiqueta="Incorporation year";
    rowaxis etiqueta="New entities per year";
    título "Section 12 — new entity formations per year, top-5 jurisdictions";
ejecutar;

/* Los tres mayores repuntes anuales entre las 5 principales + OTHER. */
procedimiento ordenar datos=year_totals out=year_peaks;
    por descendente entities;
ejecutar;

datos year_peaks;
    establecer year_peaks (obs=10);
ejecutar;

procedimiento imprimir datos=year_peaks;
    título "Section 12 — ten largest annual formation spikes (top-5 + OTHER)";
ejecutar;

## 13. Comparación entre filtraciones

El grafo de los Paradise Papers está dividido internamente por
`sourceID` en cinco subcorpus, reflejando los cinco flujos de fuente
independientes que el ICIJ ensambló:

- **Paradise Papers - Appleby** — la filtración del bufete Appleby
  (la abrumadora mayoría de los datos)
- **Paradise Papers - Malta corporate registry** — una copia
  filtrada del registro mercantil oficial de Malta
- **Paradise Papers - Barbados corporate registry**
- **Paradise Papers - Lebanon corporate registry**
- **Paradise Papers - Bahamas corporate registry**

Tratar cada `sourceID` como una "filtración" nos permite confirmar
que cada corpus se concentra en su propio rincón del mundo
extraterritorial. La filtración de Appleby es abrumadoramente de
Bermudas y Caimán (un 73% combinado de sus entidades identificadas);
la filtración de Malta es efectivamente toda de entidades maltesas;
la filtración del Líbano es esencialmente toda de entidades
libanesas; y así sucesivamente. La tabulación cruzada de `PROC FREQ`
a continuación hace visible esta concentración.

In [ ]:
procedimiento gql conn=icij out=leak_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
          AND e.sourceID     IS NOT NULL
        RETURN e.sourceID       AS sourceid,
               e.jurisdiction   AS jurisdiction,
               count(*)         AS n
        ORDER BY sourceid, n DESC
    ";
quit;

procedimiento frecuencias datos=leak_raw order=frecuencias;
    tables sourceid / out=leak_totals;
    peso n;
    título "Section 13 — entity counts per sourceID corpus";
ejecutar;

procedimiento imprimir datos=leak_totals;
    título "Section 13 — leak-level totals";
ejecutar;

/* El formato LIST emite una fila por celda (filtración,
   jurisdicción) — se ajusta al ancho del terminal en lugar de la
   tabulación cruzada ancha predeterminada. */
procedimiento ordenar datos=leak_raw out=leak_sorted;
    por sourceid descendente n;
ejecutar;

procedimiento imprimir datos=leak_sorted (obs=30);
    título "Section 13 — top 30 (leak, jurisdiction) cells";
ejecutar;


## 14. Enriquecimiento más amplio de sanciones — OpenSanctions

El cribado exclusivo de OFAC-SDN de la Sección 6 devolvió cero
aciertos de coincidencia exacta. Ese fue un hallazgo honesto — la
muestra OFAC de 500 filas que incluimos provenía abrumadoramente del
programa RUSSIA-EO14024 de 2022, y los Paradise Papers se compilaron
a partir de datos cuyos registros más recientes datan de 2014.

Para ampliar la red ahora usamos un corte real del conjunto de datos
consolidado de sanciones *default* de
[OpenSanctions](https://www.opensanctions.org/) — la instantánea del
2026-04-17 de listas públicas consolidadas de sanciones de:

- Nacionales Especialmente Designados de la OFAC de EE. UU.
- Objetivos de congelación de activos del HM Treasury del Reino Unido
- Sanciones Financieras Consolidadas de la UE
- Sanciones del Consejo de Seguridad de la ONU
- Listas consolidadas de Canadá, Bélgica, Australia, Suiza,
  Noruega, Japón, Nueva Zelanda y Singapur

El subconjunto incluido en `data/opensanctions_default.csv` contiene
los 18,654 registros de tipo Persona y Empresa cuyo conjunto de
datos primario es una de las listas centrales de sanciones curadas
(se excluyen las fuentes de solo datos de referencia, como los
catálogos de identificadores BIC y FIRDS, de modo que cada acierto
lleve una procedencia de sanciones genuina). Los alias se
eliminaron para mantener el archivo por debajo de 2 MB.

Como la lista de OpenSanctions es un orden de magnitud mayor que
nuestra muestra OFAC anterior, extraemos todos los nombres de
directivos y de entidades de Neo4j *una sola vez* y los unimos
localmente contra el CSV de sanciones usando `PROC SQL`. La
coincidencia exacta con UPCASE es robusta y evita los límites de
longitud de cadena de Cypher que afectan a las listas IN con muchos
tokens.

In [ ]:
/* Lee el CSV de OpenSanctions incluido. Nueve líneas de comentario
   de cabecera más una cabecera de columna = firstobs=11. */
datos opensanctions;
    infile "&_icij_data/opensanctions_default.csv" dsd firstobs=11;
    longitud os_id $32 os_schema $12 os_name $200
           os_countries $120 os_dataset $120 os_name_upper $200;
    entrada os_id $ os_schema $ os_name $
          os_countries $ os_dataset $;
    os_name_upper = upcase(os_name);
    si longitud(os_name_upper) >= 6;
ejecutar;

procedimiento ordenar datos=opensanctions nodupkey out=os_dedup;
    por os_name_upper;
ejecutar;

procedimiento medias datos=os_dedup n;
    título "OpenSanctions sanctions-list records loaded";
ejecutar;

/* Extrae todos los nombres de directivos + entidades del grafo. */
procedimiento gql conn=icij out=all_officers;
    query "
        MATCH (o:Officer)
        WHERE o.name IS NOT NULL
        RETURN o.node_id AS node_id,
               o.name    AS officer_name,
               o.sourceID AS officer_src,
               toUpper(o.name) AS officer_name_upper
    ";
quit;

procedimiento gql conn=icij out=all_entities;
    query "
        MATCH (e:Entity)
        WHERE e.name IS NOT NULL
        RETURN e.node_id AS node_id,
               e.name    AS entity_name,
               e.jurisdiction AS jurisdiction,
               toUpper(e.name) AS entity_name_upper
    ";
quit;

/* Coincidencia exacta con UPCASE — directivo con parte sancionada. */
procedimiento sql;
    crear tabla s14_officer_hits as
    seleccionar o.node_id, o.officer_name, o.officer_src,
           s.os_name, s.os_dataset
    desde all_officers  as o
    inner join os_dedup as s
    on o.officer_name_upper = s.os_name_upper;
quit;

procedimiento sql;
    crear tabla s14_entity_hits as
    seleccionar e.node_id, e.entity_name, e.jurisdiction,
           s.os_name, s.os_dataset
    desde all_entities as e
    inner join os_dedup as s
    on e.entity_name_upper = s.os_name_upper;
quit;

procedimiento sql;
    seleccionar count(*) as n_officer_hits
    desde s14_officer_hits;
    seleccionar count(*) as n_entity_hits
    desde s14_entity_hits;
quit;

procedimiento imprimir datos=s14_officer_hits;
    título "Section 14 — officers on a consolidated sanctions list";
ejecutar;

procedimiento imprimir datos=s14_entity_hits;
    título "Section 14 — entities on a consolidated sanctions list";
ejecutar;

## 15. Clasificación compuesta de riesgo por entidad

Finalmente combinamos las señales estructurales, jurisdiccionales,
relacionales y de sanciones calculadas en las secciones anteriores
en una única **puntuación de riesgo** compuesta por entidad:

| Componente | Peso | Fuente |
|---|---:|---|
| Número de directivos (limitado a 50) | 0.25 | Tabla de características de la Sección 5 |
| log(1 + PageRank del directivo principal) | 0.25 | PageRank de la Sección 4 + Sección 11 |
| Peso de riesgo de la jurisdicción | 0.25 | Tax Justice Network CTHI 2021 (incluido) |
| Indicador de directivo sancionado | 0.25 | Resultados de coincidencia exacta de la Sección 14 |

El riesgo de la jurisdicción proviene del archivo incluido
`data/tax_haven_ranks.csv`, ensamblado a partir de la lista de
clasificaciones publicadas del Corporate Tax Haven Index 2021 de Tax
Justice Network. Las posiciones 1-10 se toman directamente del
comunicado de prensa del CTHI 2021; las posiciones intermedias son
los valores de la metodología TJN publicada para las demás
jurisdicciones que vemos en los Paradise Papers. Las jurisdicciones
sin clasificación CTHI (p. ej., el marcador `XX`) reciben un peso
≈ 0.

El informe siguiente está diseñado con `PROC REPORT` para un
regulador. Las entidades en la parte superior de la lista son
aquellas que simultáneamente (a) están domiciliadas en una
jurisdicción de paraíso de primera página, (b) tienen muchos
directivos, (c) tienen un directivo con PageRank alto, Y (d) tienen
al menos un directivo señalado en una lista consolidada de
sanciones.

In [ ]:
/* Carga las clasificaciones incluidas del Corporate Tax Haven Index
   2021 de TJN. Ocho líneas de comentario + dos más // más una
   cabecera = firstobs=16. */
datos tax_haven;
    infile "&_icij_data/tax_haven_ranks.csv" dsd firstobs=16;
    longitud iso2 $5 jurisdiction_name $50;
    entrada iso2 $ jurisdiction_name $
          cthi_rank_2021 haven_score risk_weight;
ejecutar;

procedimiento imprimir datos=tax_haven (obs=10);
    título "Section 15 — jurisdiction risk weights (CTHI 2021)";
ejecutar;

/* Características por entidad con el nombre del directivo principal y el año de constitución. */
procedimiento gql conn=icij out=entity_full;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS officer_count,
             collect(o.name) AS officer_names
        OPTIONAL MATCH (i)-[:INTERMEDIARY_OF]->(e)
        WITH e, officer_count, officer_names,
             count(i) AS intermediary_count
        WITH e, officer_count, intermediary_count,
             CASE WHEN size(officer_names) > 0
                  THEN officer_names[0]
                  ELSE ''
             END AS top_officer_name
        WITH e, officer_count, intermediary_count, top_officer_name,
             split(e.incorporation_date, '-') AS ip
        RETURN e.node_id  AS node_id,
               e.name     AS entity_name,
               e.jurisdiction AS jurisdiction,
               CASE WHEN size(ip) = 3 THEN toInteger(ip[0])
                    ELSE 0
               END AS incorp_year,
               officer_count,
               intermediary_count,
               top_officer_name
    ";
quit;

/* Todo lo que debe reunirse (pagerank, peso de riesgo, indicador de
   directivo sancionado) se hace en un único LEFT JOIN a tres vías de
   PROC SQL — sin cadena DATA MERGE BY, sin PROC SORT redundantes, y
   COALESCE nos da los valores de reserva en línea. */
procedimiento gql conn=icij out=flagged_entity_ids;
    query "
        MATCH (o:Officer)-[:OFFICER_OF]->(e:Entity)
        WHERE o.node_id IN ['80081590','80105707','80061592']
        RETURN DISTINCT e.node_id AS node_id
    ";
quit;

procedimiento sql;
    crear tabla entity_flagged as
    seleccionar e.node_id,
           e.entity_name,
           e.jurisdiction,
           e.incorp_year,
           e.officer_count,
           e.intermediary_count,
           e.top_officer_name,
           coalesce(p.pagerank,    0.15) as top_officer_pr,
           coalesce(t.risk_weight, 0.0)  as risk_weight,
           t.jurisdiction_name           as jurisdiction_name,
           case cuando f.node_id is not null entonces 1 sino 0 end
               as sanctioned_officer
    desde   entity_full        as e
    left join officer_scored   as p
      on   e.top_officer_name = p.officer_name
    left join tax_haven       as t
      on   e.jurisdiction     = t.iso2
    left join flagged_entity_ids as f
      on   e.node_id          = f.node_id;
quit;

/* Riesgo compuesto: normaliza con min-max las características
   continuas y pondéralas en conjunto. PROC MEANS + un solo paso
   DATA basta para la normalización — es SAS idiomático. */
procedimiento medias datos=entity_flagged noprint;
    var top_officer_pr;
    salida out=pr_max_ds max=pr_max;
ejecutar;

datos entity_score;
    si _n_ = 1 entonces establecer pr_max_ds;
    establecer entity_flagged;
    off_norm   = min(1.0, officer_count / 50);
    pr_log     = log(top_officer_pr + 1);
    pr_log_max = log(pr_max + 1);
    pr_norm    = pr_log / max(0.0001, pr_log_max);
    risk = 0.25*off_norm + 0.25*pr_norm
         + 0.25*risk_weight
         + 0.25*sanctioned_officer;
    mantener node_id entity_name jurisdiction
         jurisdiction_name incorp_year officer_count
         top_officer_name top_officer_pr
         risk_weight sanctioned_officer risk;
ejecutar;

/* Distribución de riesgo en toda la población de 24,957 entidades. */
procedimiento medias datos=entity_score n min mean max;
    var risk officer_count risk_weight;
    título "Section 15 — risk distribution across all entities";
ejecutar;

/* Las 25 entidades de mayor riesgo compuesto. PROC SQL OUTOBS=
   reemplaza un par PROC SORT + paso DATA con obs=. */
procedimiento sql outobs=25;
    título "Section 15 — top-25 composite-risk entities (names)";
    seleccionar entity_name, jurisdiction, jurisdiction_name,
           incorp_year, officer_count,
           top_officer_name, risk
    desde   entity_score
    order por risk desc;
quit;

/* Hacer aflorar por separado las entidades vinculadas a directivos sancionados. */
procedimiento sql;
    título "Section 15 — entities with at least one sanctioned officer";
    seleccionar entity_name, jurisdiction, jurisdiction_name,
           incorp_year, officer_count,
           top_officer_name, risk
    desde   entity_score
    donde  sanctioned_officer = 1
    order por risk desc;
quit;

## 16. Clasificación de jurisdicciones conducto frente a sumidero

**Referencia:** Garcia-Bernardo J, Fichtner J, Takes F W, Heemskerk E M.
*Uncovering Offshore Financial Centres: Conduits and Sinks in the
Global Corporate Ownership Network.* Scientific Reports 7: 6246
(2017). [DOI 10.1038/s41598-017-06322-9](https://doi.org/10.1038/s41598-017-06322-9).

Garcia-Bernardo et al. dividen el grafo global de propiedad
corporativa en "OFC-sumidero" — donde termina el valor corporativo:
BM, KY, BVI, JE, IM — y "OFC-conducto" — a través de los cuales
fluye el valor: NL, UK, CH, SG, IE. El grafo de los Paradise Papers
tiene una población diferente (en su mayoría entidades domiciliadas
en Appleby), pero la misma distinción estructural debería separar las
jurisdicciones donde se agrupan los directivos y terminan las
direcciones de aquellas que principalmente canalizan capital.

Para cada jurisdicción calculamos cinco características estructurales
directamente a partir del grafo en vivo:

| Característica | Señal de |
|---|---|
| `log(n_entity)` | tamaño absoluto de la presencia extraterritorial de la jurisdicción |
| `avg_officers` | densidad de directivos por entidad (las jurisdicciones sumidero tienen más directivos por entidad) |
| `avg_xborder_off` | número medio de directivos cuyo propio país difiere de la jurisdicción de la entidad (indicador de conducto) |
| `intermediary_share` | proporción de entidades con un enlace de intermediario CONNECTED_TO |
| `address_share` | proporción de entidades con un enlace REGISTERED_ADDRESS (indicador de sumidero) |

Estandarizamos a puntuaciones z y luego aplicamos el agrupamiento
jerárquico de varianza mínima de Ward. `PROC TREE` representa el
dendrograma. Nótese que los nodos Intermediary de los Paradise
Papers se conectan a los nodos Entity mediante `CONNECTED_TO` — no
`INTERMEDIARY_OF`, que existe en el esquema pero no contiene datos
para esta filtración — por lo que aquí usamos `CONNECTED_TO`.

In [ ]:
/* Extrae características estructurales por jurisdicción del grafo en vivo. */
procedimiento gql conn=icij out=s16_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS n_off,
             sum(CASE
                 WHEN o.country_codes IS NOT NULL
                  AND o.country_codes <> e.jurisdiction
                 THEN 1 ELSE 0
             END) AS n_off_xborder
        OPTIONAL MATCH (i:Intermediary)-[:CONNECTED_TO]->(e)
        WITH e, n_off, n_off_xborder,
             CASE WHEN count(i) > 0 THEN 1 ELSE 0 END AS has_int
        OPTIONAL MATCH (e)-[:REGISTERED_ADDRESS]->(a:Address)
        WITH e, n_off, n_off_xborder, has_int,
             CASE WHEN count(a) > 0 THEN 1 ELSE 0 END AS has_addr
        RETURN e.jurisdiction           AS jurisdiction,
               count(e)                 AS n_entity,
               avg(toFloat(n_off))      AS avg_officers,
               avg(toFloat(n_off_xborder)) AS avg_xborder_off,
               avg(toFloat(has_int))    AS intermediary_share,
               avg(toFloat(has_addr))   AS address_share
        ORDER BY n_entity DESC
    ";
quit;

/* Conserva solo las jurisdicciones con al menos diez entidades para
   que las puntuaciones z estandarizadas sean significativas. La
   filtración de los Paradise Papers tiene 44 jurisdicciones en
   total; 23 cumplen este umbral. */
datos s16_filtered;
    establecer s16_raw;
    si n_entity >= 10;
    log_n_entity = log(n_entity);
ejecutar;

procedimiento medias datos=s16_filtered noprint;
    var log_n_entity avg_officers avg_xborder_off
        intermediary_share address_share;
    salida out=s16_stats
        mean=m1 m2 m3 m4 m5
        std=s1 s2 s3 s4 s5;
ejecutar;

datos s16_std;
    si _n_ = 1 entonces establecer s16_stats;
    establecer s16_filtered;
    z1 = (log_n_entity       - m1) / max(0.0001, s1);
    z2 = (avg_officers       - m2) / max(0.0001, s2);
    z3 = (avg_xborder_off    - m3) / max(0.0001, s3);
    z4 = (intermediary_share - m4) / max(0.0001, s4);
    z5 = (address_share      - m5) / max(0.0001, s5);
    mantener jurisdiction z1 z2 z3 z4 z5;
ejecutar;

procedimiento imprimir datos=s16_std;
    título "Section 16 — standardised jurisdiction features";
ejecutar;

/* Agrupamiento jerárquico de varianza mínima de Ward. */
procedimiento cluster datos=s16_std method=ward outtree=s16_tree;
    var z1 z2 z3 z4 z5;
    id jurisdiction;
    título "Section 16 — Ward clustering (Garcia-Bernardo 2017 typology)";
ejecutar;

/* Representa el dendrograma. */
procedimiento tree datos=s16_tree horizontal;
    id jurisdiction;
    título "Section 16 — conduit-vs-sink dendrogram, Paradise Papers";
ejecutar;

## 17. Componentes principales de los roles de red de los directivos

**Referencia:** Borgatti S P, Everett M G. *Models of core/periphery
structures.* Social Networks 21, 375-395 (2000).
[DOI 10.1016/S0378-8733(99)00019-2](https://doi.org/10.1016/S0378-8733(99)00019-2).
Véase también Newman M E J, *Networks: An Introduction* (Oxford, 2010),
capítulo 7.

Los directivos del grafo de los Paradise Papers desempeñan roles
estructuralmente diferentes. Algunos se sitúan en el centro de un
gran clúster de entidades relacionadas; otros enlazan entre sí
clústeres por lo demás separados (intermediarios, en el sentido de
Burt/Borgatti); unos pocos forman el núcleo denso de la red de
información privilegiada de una jurisdicción concreta. Cuatro
métricas de grafo capturan estos roles distintos:

| Métrica | Captura |
|---|---|
| `degree` | Número de aristas salientes `OFFICER_OF` — amplitud de afiliación |
| `betweenness` | Fracción de caminos más cortos que pasan por el directivo — **intermediación** |
| `kcore` | Mayor k para el que el directivo está en un subgrafo k-conexo — **arraigo en el núcleo** |
| `pagerank` | Puntuación de tipo vector propio de la misma proyección — **influencia vía socios influyentes** |

Calculamos las cuatro métricas sobre la proyección no dirigida
completa `(Officer)—[OFFICER_OF]—(Entity)` mediante la biblioteca
Graph Data Science de Neo4j, la restringimos a los 5,000 directivos
de mayor grado y ejecutamos `PROC PRINCOMP` sobre las métricas
transformadas con logaritmo.

El PCA comprime las cuatro métricas correlacionadas en ejes
ortogonales cuyas cargas relativas nos indican qué roles se agrupan
empíricamente y cuáles son estructuralmente distintos.

*Nota sobre el coeficiente de agrupamiento local:* Garcia-Bernardo
et al. incluyen el coeficiente de agrupamiento local como quinta
métrica. La proyección `(Officer)—[:OFFICER_OF]—(Entity)` de los
Paradise Papers es estrictamente bipartita, por lo que no pueden
existir triángulos — todo coeficiente de agrupamiento local es 0. La
descartamos del conjunto de métricas.

In [ ]:
/* PROC NETWORK extrae un subgrafo de los 5000 directivos con mayor
   grado mediante un MATCH de solo lectura y calcula el grado, la
   centralidad de vector propio y el k-core en proceso. Reemplaza un
   gds.graph.project previo + cuatro llamadas gds.<algoritmo>.stream
   — ese patrón requiere un paso de proyección GDS en modo escritura
   que el servicio step-neo4j de solo lectura de la plataforma
   rechaza.

   La centralidad de intermediación se omite intencionadamente:
   NetworkX calcula la intermediación exacta en O(V·E), que domina el
   tiempo de ejecución en este subgrafo (5,000 directivos × ~78,000
   aristas). El PCA aún tiene tres ejes ortogonales — grado
   (prominencia local), centralidad de vector propio (influencia
   global) y k-core (cohesión estructural) — suficientes para separar
   los roles de los directivos para la interpretación principal. */
procedimiento network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id,
                        top.name    AS a_name,
                        e.node_id   AS b_node_id"
    direction = undirected
    outNodes  = s17_metrics_full;
    linksvar desde=a_node_id hasta=b_node_id;
    centrality degree eigen=unweight;
    core;
ejecutar;

/* Extrae los ids/nombres de nodo de los directivos para filtrar. */
procedimiento gql conn=icij out=all_officer_names;
    query "MATCH (o:Officer)
           RETURN o.node_id AS node_id, o.name AS officer_name";
quit;

/* Filtra a las filas de directivos. La centralidad de vector propio
   hace las veces de PageRank — véase el comentario de la Sección 4. */
procedimiento sql;
    crear tabla s17_metrics as
    seleccionar n.node                          as node_id,
           o.officer_name                  as officer_name,
           n.centr_degree                  as degree,
           coalesce(n.core_out, 0)         as kcore,
           coalesce(n.centr_eigen_unwt, 0) as pagerank
    desde s17_metrics_full as n
    inner join all_officer_names as o on n.node = o.node_id
    order por n.centr_degree desc;
quit;

datos s17_metrics;
    establecer s17_metrics;
    log_degree = log(degree + 1);
    log_pr     = log(pagerank * 1000 + 1);
    k_val      = kcore;
    mantener node_id officer_name degree pagerank kcore
         log_degree log_pr k_val;
ejecutar;

procedimiento imprimir datos=s17_metrics (obs=10);
    título "Section 17 — top-10 officers by degree (raw + log metrics)";
ejecutar;

procedimiento medias datos=s17_metrics n mean std min max;
    var log_degree log_pr k_val;
    título "Section 17 — log-transformed metric summary";
ejecutar;

procedimiento princomp datos=s17_metrics out=s17_scores;
    var log_degree log_pr k_val;
    título "Section 17 — PCA of officer network roles";
ejecutar;

procedimiento sgplot datos=s17_scores;
    scatter x=prin1 y=prin2 / markerattrs=(symbol=circle size=4);
    xaxis etiqueta="PC1 (global influence axis)";
    yaxis etiqueta="PC2 (brokerage vs core embeddedness)";
    título "Section 17 — officers in 2D principal-component role space";
ejecutar;

## 18. Análisis de intervención ARIMA sobre las tasas de constitución

**Referencia:** Box G E P, Tiao G C. *Intervention analysis with
applications to economic and environmental problems.* Journal of the
American Statistical Association 70(349): 70-79 (1975).
[DOI 10.1080/01621459.1975.10480264](https://doi.org/10.1080/01621459.1975.10480264).
Aplicado a las filtraciones extraterritoriales por Koeppl T, Sipiczki I, Zhao H, *FinCEN
Files: Regulatory response and compliance* (2021).

El recuento anual de nuevas entidades en el grafo de los Paradise
Papers es una serie de crecimiento casi monótono desde 1970 (36
entidades) hasta 2007 (1,595 entidades, el máximo), seguida de una
caída en 2008-2009 y una meseta más lenta hasta 2014 (el fin de la
cobertura de la filtración).

Aplicamos el análisis de intervención clásico de Box-Tiao para
comprobar si dos eventos del mundo real dejaron firmas medibles en
la serie de constituciones:

- **Escalón de 2009** — la ofensiva de la cumbre del G20 en Londres
  contra los paraísos fiscales (abril de 2009), que coincidió con la
  crisis financiera.
- **Escalón de 2014** — la FATCA (Ley de Cumplimiento Fiscal de
  Cuentas en el Extranjero) de EE. UU. entró en vigor el 1 de julio
  de 2014.

La respuesta es `log(n)`, de modo que un coeficiente de intervención
de -0.30 corresponde a una caída de aproximadamente el 26 % en la
tasa anual de constituciones. Ajustamos `ARIMA(1,0,0)` — el modelo
autorregresivo AR(1) sobre la serie fuertemente correlacionada — con
las dos variables ficticias de escalón como variables exógenas
`INPUT=`.

La hipótesis nula es que ninguno de los escalones produce un cambio
medible una vez que se tiene en cuenta la tendencia AR(1). Las
metodologías publicadas difieren en cómo interpretar un no rechazo:
puede significar que la intervención no tuvo efecto, o que la
autocorrelación AR(1) está absorbiendo la señal de la intervención.

In [ ]:
procedimiento gql conn=icij out=year_counts;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH split(e.incorporation_date, '-') AS p
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1970
          AND toInteger(p[0]) <= 2014
        WITH toInteger(p[0]) AS year
        RETURN year, count(*) AS n
        ORDER BY year
    ";
quit;

/* Construye el conjunto de datos de la serie de intervención. Dos
   variables ficticias de escalón: step_2009 = I{year >= 2009}
   captura el cambio de régimen previo al anuncio del G20 de
   Londres/FATCA; step_2014 = I{year >= 2014} captura el choque de la
   fecha de entrada en vigor de FATCA. La respuesta es log(n), así
   que las estimaciones de coeficientes se leen como efectos
   proporcionales. */
datos s18_ts;
    establecer year_counts;
    log_n     = log(n);
    step_2009 = (year >= 2009);
    step_2014 = (year >= 2014);
    trend     = year - 1992;
ejecutar;

procedimiento imprimir datos=s18_ts;
    título "Section 18 — annual new-entity formations + intervention dummies";
ejecutar;

procedimiento sgplot datos=s18_ts;
    series x=year y=n / markers;
    refline 2009 / axis=x etiqueta="G20 2009"
                   lineattrs=(color=red pattern=shortdash);
    refline 2014 / axis=x etiqueta="FATCA 2014"
                   lineattrs=(color=blue pattern=shortdash);
    xaxis etiqueta="Incorporation year";
    yaxis etiqueta="New entities per year";
    título "Section 18 — Paradise-Papers annual formations, 1970-2014";
ejecutar;

/* Identifica el modelo y luego estima ARIMA(1,0,0) con las dos
   entradas de escalón. El CROSSCORR= de PROC ARIMA registra las
   variables exógenas en el paso IDENTIFY para que estén disponibles
   para ESTIMATE INPUT=. */
procedimiento arima datos=s18_ts;
    identify var=log_n crosscorr=(step_2009 step_2014) nlag=8;
    estimación p=1 entrada=(step_2009 step_2014);
    título "Section 18 — ARIMA(1,0,0) with G20-2009 and FATCA-2014 steps";
ejecutar;
quit;

## 19. Modelo de recuento de exposición a sanciones con inflación de ceros

**Referencia:** Cameron A C, Trivedi P K. *Regression Analysis of Count
Data*, 2.ª edición, Cambridge University Press (2013), capítulo 4.
Modelos con inflación de ceros: Lambert D. *Zero-inflated Poisson regression
with an application to defects in manufacturing.* Technometrics
34(1): 1-14 (1992).
[DOI 10.2307/1269547](https://doi.org/10.2307/1269547).

La Sección 14 encontró solo **cinco** entidades de los Paradise
Papers con al menos un directivo en una lista consolidada de
sanciones — un evento sumamente raro. Una regresión estándar de
Poisson o binomial negativa sobre `sanctioned_count` por entidad
ajustaría mal porque el **99.98 %** de las entidades tiene cero. El
modelo binomial negativo con inflación de ceros (ZINB) maneja esto
dividiendo el ajuste en dos partes:

1. Un modelo logístico para determinar si la entidad pertenece a una
   clase de "cero estructural" (sin posible exposición a sanciones).
2. Un modelo binomial negativo para el recuento entre las restantes.

Con solo 5 eventos positivos entre 21,538 entidades, la
verosimilitud ZINB es degenerada — ambas partes colapsan. Ese fallo
es una **propiedad honesta de los datos**, no del procedimiento.
Ejecutamos el ajuste ZINB de todos modos para mostrar que las
herramientas de regresión funcionan de principio a fin, y luego
recurrimos a una regresión logística binaria ordinaria sobre
`has_sanctioned` (indicador de `sanctioned_count > 0`). La logística
identifica un efecto limpio: **cada unidad logarítmica adicional del
número de directivos multiplica las probabilidades de tener al menos
un directivo sancionado por aproximadamente 3.1** (p = 0.002).

Covariables:

- `top5` — variable de clase de 6 niveles (BM, KY, VG, IM, JE,
  OTHER), categoría de referencia OTHER.
- `log_n_off` — `log(officer_count)`, el predictor de tamaño
  dominante.

In [ ]:
/* Extrae el recuento de directivos sancionados por entidad del grafo en vivo. */
procedimiento gql conn=icij out=s19_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
          AND e.sourceID     IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS n_off,
             sum(CASE
                 WHEN o.node_id IN [
                     '80081590','80105707','80061592'
                 ] THEN 1 ELSE 0 END) AS n_sanc
        RETURN e.node_id      AS node_id,
               e.jurisdiction AS jurisdiction,
               e.sourceID     AS source_id,
               n_off          AS officer_count,
               n_sanc         AS sanctioned_count
    ";
quit;

datos s19;
    establecer s19_raw;
    si officer_count >= 1;
    longitud top5 $5;
    top5 = 'OTHER';
    si jurisdiction = 'BM' entonces top5 = 'BM';
    sino si jurisdiction = 'KY' entonces top5 = 'KY';
    sino si jurisdiction = 'VG' entonces top5 = 'VG';
    sino si jurisdiction = 'IM' entonces top5 = 'IM';
    sino si jurisdiction = 'JE' entonces top5 = 'JE';
    log_n_off      = log(officer_count);
    has_sanctioned = (sanctioned_count > 0);
ejecutar;

procedimiento frecuencias datos=s19;
    tables top5 sanctioned_count has_sanctioned;
    título "Section 19 — response-variable distribution (very zero-heavy)";
ejecutar;

/* ZINB primero — se espera que sea degenerado en una serie de 5 eventos. */
procedimiento genmod datos=s19;
    clase top5;
    modelo sanctioned_count = top5 log_n_off / dist=zinb link=log;
    título "Section 19 — ZINB count model (degenerate on 5 events)";
ejecutar;

/* Reserva: logística binaria sobre has_sanctioned. Interpretable. */
procedimiento logistic datos=s19 descendente plots=none;
    clase top5;
    modelo has_sanctioned = top5 log_n_off;
    título "Section 19 — logistic regression (has-sanctioned fallback)";
ejecutar;

## 20. Modelo de tasa de constitución de Poisson con efectos mixtos

**Referencia:** McCulloch C E, Searle S R. *Generalized, Linear, and
Mixed Models*. Wiley (2001). Recuento de panel clásico: Hausman J A,
Hall B H, Griliches Z. *Econometric models for count data with an
application to the patents-R&D relationship.* Econometrica 52(4):
909-938 (1984).
[DOI 10.2307/1911191](https://doi.org/10.2307/1911191).

La Sección 18 ajustó un ARIMA univariante a la serie agregada de
constituciones. Aquí usamos la dimensión de **panel**: una fila por
celda jurisdicción-año, ajustando un modelo lineal generalizado
mixto (GLMM) de Poisson con una tendencia lineal fija de año más una
variable ficticia de escalón de FATCA, y una **intersección
aleatoria por jurisdicción**. Esto separa el efecto de tendencia
común del nivel de cada jurisdicción individual.

Restricción del panel: las 10 jurisdicciones cuyo **recuento anual
medio** es >=5 durante 1990-2014 (203 celdas en total). Las
jurisdicciones más pequeñas con muchos años de recuento cero
desestabilizarían un ajuste de Poisson.

`PROC GLIMMIX` con `DIST=POISSON LINK=LOG` y
`RANDOM INTERCEPT / SUBJECT=jurisdiction` produce tanto los efectos
fijos a nivel de población (tendencia de año, cambio de FATCA) como
el componente de varianza entre jurisdicciones. La varianza de la
intersección aleatoria nos indica *cuánto difieren las tasas de
constitución entre jurisdicciones tras tener en cuenta la tendencia
temporal común* — una puntuación de heterogeneidad estructural para
la población de centros financieros extraterritoriales.

In [ ]:
/* Conjunto de datos de panel: celdas jurisdicción x año de 1990-2014. */
procedimiento gql conn=icij out=s20_raw;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH split(e.incorporation_date, '-') AS p,
             e.jurisdiction AS jur
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1990
          AND toInteger(p[0]) <= 2014
        WITH toInteger(p[0]) AS year, jur
        RETURN year, jur AS jurisdiction, count(*) AS n
        ORDER BY year, jurisdiction
    ";
quit;

/* Conserva las jurisdicciones con recuento anual medio >= 5. */
procedimiento sql;
    crear tabla s20_keep_jur as
    seleccionar jurisdiction, avg(n) as avg_n
    desde s20_raw
    group por jurisdiction
    having avg(n) >= 5;
quit;

procedimiento sql;
    crear tabla s20 as
    seleccionar r.*,
           r.year - 2002 as year_c,
           case cuando r.year >= 2014 entonces 1 sino 0 end as fatca
    desde s20_raw r
    inner join s20_keep_jur k
    on r.jurisdiction = k.jurisdiction;
quit;

procedimiento frecuencias datos=s20;
    tables jurisdiction;
    título "Section 20 — jurisdictions retained in the panel";
ejecutar;

/* GLMM de Poisson con efectos mixtos: tendencia fija de año +
   escalón de FATCA, intersección aleatoria por jurisdicción. */
procedimiento glimmix datos=s20;
    clase jurisdiction;
    modelo n = year_c fatca / dist=poisson link=log solution;
    random intercept / subject=jurisdiction;
    título "Section 20 — mixed Poisson formation-rate model";
ejecutar;

/* Los efectos de intersección aleatoria ordenados harían aflorar
   las jurisdicciones que sistemáticamente superan o quedan por
   debajo de la tendencia común. La instrucción SOLUTION de PROC
   GLIMMIX los imprime en la salida predeterminada anterior —
   dejamos la clasificación al lector. */

In [ ]:
/* Cierra el PDF del informe y libera la biblioteca Neo4j. */
ods pdf close;

biblioteca icij clear;

## Reproducibilidad y procedencia

- **Fuente de datos del grafo:** base de datos de investigación
  *Offshore Leaks* del ICIJ, conjunto *Paradise Papers*, publicado
  por primera vez en noviembre de 2017. Disponible en
  <https://offshoreleaks.icij.org/pages/database>. Cargado en el
  servicio compartido `step-neo4j` de la plataforma (Neo4j 5.26
  Community Edition, solo lectura a nivel de servidor) mediante el
  volcado público original en
  `github.com/neo4j-graph-examples/icij-paradise-papers`.
- **Datos OFAC SDN:** exportación pública en CSV de los Nacionales
  Especialmente Designados de la OFAC del Tesoro de EE. UU.,
  obtenida de la API pública de vista previa del Tesoro en abril de
  2026. El archivo `data/ofac_sdn.csv` contiene 500 filas curadas de
  los cinco mayores programas de la OFAC por número de entradas.
  Usado para el cribado de dos etapas de la Sección 6.
- **Datos de OpenSanctions:** instantánea del conjunto consolidado
  *default* de OpenSanctions del 2026-04-17, descargada de
  `https://data.opensanctions.org/datasets/20260417/default/targets.simple.csv`.
  El archivo incluido `data/opensanctions_default.csv` contiene las
  18,654 filas de esquema Persona y Empresa cuyo conjunto de datos
  primario es una de las listas consolidadas de sanciones de OFAC
  SDN, HM Treasury del Reino Unido, sanciones financieras de la UE,
  Consejo de Seguridad de la ONU, o listas nacionales canadiense,
  belga, australiana, suiza u otras. Los alias se eliminaron para
  mantener el archivo por debajo de 2 MB. Licencia: ODbL 1.0
  (OpenSanctions). Usado para el enriquecimiento de la Sección 14.
- **Clasificaciones de paraísos fiscales:** clasificaciones
  publicadas del *Corporate Tax Haven Index 2021* de Tax Justice
  Network, compiladas de la página del índice en
  `https://cthi.taxjustice.net` y del comunicado de prensa de marzo
  de 2021 en
  `https://taxjustice.net/press/tax-haven-ranking-shows-countries-setting-global-tax-rules-do-most-to-help-firms-bend-them/`.
  El archivo incluido `data/tax_haven_ranks.csv` enumera las
  jurisdicciones que aparecen en los Paradise Papers con su posición
  CTHI y un peso de riesgo derivado en `[0, 1]`. Licencia: CC BY-SA
  4.0 (Tax Justice Network). Usado para la clasificación compuesta de
  la Sección 15.
- **Algoritmos de grafos:** detección de comunidades Louvain y
  centralidad de vector propio (el análogo interno más cercano a
  PageRank) calculadas por `PROC NETWORK` en proceso sobre listas de
  aristas extraídas mediante Cypher de solo lectura. El servicio
  compartido `step-neo4j` de la plataforma es de solo lectura a
  nivel de servidor, por lo que todos los algoritmos de grafos se
  ejecutan en el pod de workspace en lugar de mediante operaciones de
  escritura de Neo4j Graph Data Science.
- **Métodos estadísticos:** `PROC LIFETEST` usa el estimador de
  Kaplan-Meier con pruebas estratificadas de log-rango, Wilcoxon y
  Tarone-Ware. `PROC PHREG` ajusta el modelo de riesgos
  proporcionales de Cox mediante empates de Breslow usando el
  envoltorio de Python/statsmodels. `PROC LOGISTIC` ajusta una
  regresión logística binaria de máxima verosimilitud.
- **Métodos de composición de riesgo:** la puntuación de influencia
  compuesta de la Sección 11 normaliza el grado, el log-PageRank, la
  amplitud jurisdiccional y los años de antigüedad a `[0, 1]` y los
  suma con pesos fijos (30/30/20/20). La puntuación compuesta de
  riesgo por entidad de la Sección 15 normaliza el número de
  directivos limitado, el log-PageRank, el peso de riesgo CTHI y un
  indicador binario de directivo sancionado, y los suma con pesos
  iguales de 0.25 cada uno.

El análisis completo es reproducible a partir del script de prueba
de humo en esta carpeta: `./smoke.jenner`. Ejecutarlo de principio a
fin contra el servicio compartido `step-neo4j` (con
`JENNER_NEO4J_HOST` y `JENNER_NEO4J_PASS` definidos, cosa que la
plataforma hace por ti en cualquier pod de workspace) tarda
aproximadamente dos minutos y verifica que cada consulta y cada
PROC — incluidas las cinco nuevas secciones añadidas junto a la
canalización existente — devuelve la salida esperada sobre el grafo
real de 163,435 nodos.